# solar-flare-utac — Package 21 Overview

**GenesisAeon Package 21** — Solar Flare Magnetic Avalanche Threshold

## Central Result

> Γ_solar = arctanh(0.03) / 2.2 ≈ **0.014**

Solar active regions occupy the **ultra-sensitive** end of the CREP criticality spectrum.
Even tiny CREP fluctuations translate into giant energy releases — explaining why
deterministic solar flare prediction remains notoriously difficult.

## Diamond-Template Contract

```python
model = SolarFlareUTAC(seed=42)
state  = model.run_cycle(duration_hours=72)
crep   = model.get_crep_state()    # {C, R, E, P, Gamma}
utac   = model.get_utac_state()    # {H, dH_dt, H_star, K_eff}
events = model.get_phase_events()  # list of flare dicts
record = model.to_zenodo_record()  # Zenodo metadata
```

## Physical Mapping

| UTAC symbol | Solar quantity |
|---|---|
| H(t) | Normalised magnetic free energy ∈ [0, 1] |
| K | E_max ≈ 10³² J (active region ceiling) |
| H* | Quiescent fixed point ≈ 0.10 K |
| r | Flux-emergence buildup rate ≈ 0.06 hr⁻¹ |
| σ | CREP coupling = 2.2 |
| Γ(t) | CREP tensor (C, R, E, P) → Γ ≈ 0.014 |

In [ ]:
from solar_flare_utac import SolarFlareUTAC, GAMMA_SOLAR
import matplotlib.pyplot as plt
import numpy as np

print(f'Γ_solar (nominal) = {GAMMA_SOLAR:.5f}')

In [ ]:
# Run a 72-hour UTAC cycle
model = SolarFlareUTAC(seed=42)
state = model.run_cycle(duration_hours=72.0)

print('UTAC state:', state['utac_state'])
print('CREP state:', state['crep_state'])
print('Flares this cycle:', state['flare_count'])

In [ ]:
# Simulate active region energy evolution
from solar_flare_utac.active_region import MagneticActiveRegion
from solar_flare_utac.constants import R_BUILDUP, H_STAR_QUIET

ar = MagneticActiveRegion(r_buildup=R_BUILDUP, h_init=H_STAR_QUIET, seed=42)
result = ar.simulate(duration_hours=96.0, dt_hours=0.1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(result['times'], result['H'], lw=0.8, color='orangered')
axes[0].axhline(0.60, ls='--', color='red', label='H_threshold')
axes[0].axhline(0.10, ls='--', color='blue', label='H* quiet')
axes[0].set_xlabel('Time [hours]')
axes[0].set_ylabel('H (normalised free energy)')
axes[0].set_title('Active Region Energy Evolution')
axes[0].legend()

# Mark flare events
for evt in result['flare_events']:
    axes[0].axvline(evt['time_h'], color='gold', alpha=0.7, lw=1)

# Flare energy histogram
if result['flare_events']:
    energies = [e['energy_J'] for e in result['flare_events']]
    axes[1].hist(energies, bins=20, color='orangered', edgecolor='black')
    axes[1].set_xlabel('Flare Energy [J]')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Flare Energy Distribution (N={len(energies)})')

plt.tight_layout()
plt.savefig('../data/fig_01_overview.png', dpi=100)
plt.show()

In [ ]:
# CREP Criticality Spectrum position
spectrum = [
    ('Solar Flare',         0.014, 'P21'),
    ('Cygnus X-1 Jet',     0.046, 'P17'),
    ('Amazon Rainforest',  0.116, 'P19'),
    ('AMOC Ocean',         0.251, 'P18'),
    ('Neural Criticality', 0.251, 'P20'),
    ('BTW Sandpile',       0.296, 'P22'),
    ('Manna Sandpile',     0.376, 'P22'),
    ('ERA5 Arctic',        0.920, 'P01'),
]

names, gammas, pkgs = zip(*spectrum)
colours = ['red' if p == 'P21' else 'steelblue' for p in pkgs]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(names, gammas, color=colours)
ax.set_xlabel('Γ (CREP coupling)')
ax.set_title('GenesisAeon CREP Criticality Spectrum (Packages 17–22)')
ax.axvline(0.014, ls='--', color='red', label='Γ_solar ≈ 0.014')
ax.legend()
plt.tight_layout()
plt.show()